In [0]:
import json
import os
from urllib.parse import urljoin
import pandas as pd
import requests

BASE_URL = 'https://cvijan.github.io/nss-api/v2/'
FILES_URL = 'https://cvijan.github.io/nss-api/files/'
RAW_DIR = 'data/raw'
PROCESSED_DIR = 'data/processed'

In [0]:
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

In [0]:
# 1
def get_json(url: str) -> dict:
    response = requests.get(url, timeout = 10)
    response.raise_for_status()
    return response.json()

In [0]:
# 2
def fetch_all(first_url: str) -> list:
    records = []
    url = first_url

    while url:
        page = get_json(url)
        records.extend(page['data'])

        if page.get('next'):
            url = urljoin(url, page['next'])
        else:
            url = None
    return records

orders = fetch_all( BASE_URL + 'orders/page-1.json')
print(len(orders))

1852


In [0]:
# 3
def save_json(records: list, path: str) -> None:
    with open(path, 'w') as f:
        json.dump(records, f, ensure_ascii=False, indent=2)

save_json(orders, os.path.join(RAW_DIR, 'orders.json'))


In [0]:
# 4
def download_file(url: str, path: str) -> int:
    response = requests.get(url, timeout=30)
    response.raise_for_status() # 404 and other errors stop here
    with open(path, 'wb') as f: # 'wb' = write bytes, needed for Parquet
        f.write(response.content) # .content = the file's bytes, unchanged
    return len(response.content)


8022

In [0]:
# 5
def extract(raw_dir: str) -> dict:
    orders = fetch_all(BASE_URL + 'orders/page-1.json')

    download_file(FILES_URL + 'products.csv', os.path.join(RAW_DIR,'products.csv'))
    download_file(FILES_URL + 'teams.csv', os.path.join(RAW_DIR,'teams.csv'))
    download_file(FILES_URL + 'matches.parquet', os.path.join(RAW_DIR,'matches.parquet'))

    orders_df = pd.json_normalize(orders, sep='_')
    orders_df = orders_df.rename( columns = { 'customer_name':'customer',
                                             'customer_city':'city'})
    products_df = pd.read_csv(os.path.join(RAW_DIR, 'products.csv'))
    teams_df = pd.read_csv(os.path.join(RAW_DIR, 'teams.csv'))
    matches_df = pd.read_parquet(os.path.join(RAW_DIR, 'matches.parquet'))

    return {
        'orders' : orders_df,
        'products' : products_df,
        'teams' : teams_df,
        'matches' : matches_df
    }
data = extract(RAW_DIR)

for name, df in data.items():
    print(name, df.shape)



orders (1852, 8)
products (108, 7)
teams (20, 5)
matches (120, 9)


In [0]:
# 6

def clean_products(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df = df.remove_duplicates(subset = 'product_id', keep = 'first')

    df['team_code'] = df['team_code'].str.strip().upper()
    df['category'] = df['category'].str.strip().lower()

    df['price'] = df['price'].str.replace(' RSD', '', regex=False).replace(',', '.', regex = False)
    df['price'] = pd.to_numeric(df['price'])
    



In [0]:
# 7
def clean_orders(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    columns = ['order_id', 'customer', 'city', 'payment']
    for col in columns:
        df[col] = df[col].strip()

    df['order_id'] = df['order_id'].str.upper()
    df['customer'] = df['customer'].str.title()
    df['city'] = df['city'].str.title()
    df['city'] = df['city'].str.fillna('Unknown')

    return df



In [0]:
# 8
def remove_duplicate_orders(df: pd.DataFrame) -> tuple:
    duplicates = df[df.duplicates(subset = 'order_id')]
    unique = df[df.remove_duplicates(subset = 'order_id', keep = 'first')]

    return duplicates, unique



In [0]:
# 9
def validate_order(row: dict, product_ids: set) -> str:

    if not row['order_id'].startswith('NSS'):
        return 'invalid_order'
    if pd.isna(row['quantity']):
        return 'missing_quantity'
    if pd.to_numeric(row['quantity'], errors='coerce') > 0:
        return 'invalid_quantity'
    if pd.to_numeric(row['unit_price']) > 0:
        return 'invalid_price'
    if row['product_id'] not in product_ids:
        return ' unknown_product'
    return 'OK'

In [0]:
# 10
def transform_orders(df: pd.DataFrame, products: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df['quantity'] = df['quantity'].astype(int)
    df['unit_price'] = df['unit_price'].astype(float)
    df['order_date'] = pd.to_datetime(df['order_date'])

    df['total'] = df['quantity'] * df['unit_price']

    df = df.merge(products[['product_id', 'product_name', 'team_code', 'category']],
                  on = 'product_id', how = 'left')
    return df


In [0]:
# 11
def split_orders(df: pd.DataFrame, products: pd.DataFrame) -> tuple:
    cleaned = clean_orders(df)
    duplicates, unique = remove_duplicate_orders(df)
    product_ids = set(products['product_id'])
    duplicates = duplicates.assign(reason = 'duplicate')
    reasons = unique.apply(lambda row: validate_order(row.to_dict(), product_ids), axis = 1)
    valid = transform_orders(unique[reasons == 'OK'], products)
    rejected = pd.concat([unique[reasons != 'OK'].assign(reason = reasons[reasons != 'OK']), duplicates])
    return valid, rejected




In [0]:
# 17
def load(outputs: dict, processed_dir: str) -> dict:
    written = {}
    for name, df in outputs.items():
        path = os.path.join(processed_dir, name)
        if name.endswith('.parquet'):
            df.to_parquet(path, index=False)
        else:
            df.to_csv(path, index=False)
        written[name] = len(df)
    return written

In [0]:
# 18
def run_pipeline(raw_dir: str, processed_dir: str) -> dict:
    data = extract(raw_dir)
    products = clean_products(data['products'])
    valid, rejected = split_orders(data['orders'], products)
    results = team_results(data['matches'], data['teams'])
    windows = match_windows(results)
    outputs = {
        'clean_orders.parquet': valid,
        'rejected_orders.csv': rejected,
        'products.csv': products,
        'league_table.csv': league_table(results),
        'revenue_by_team.csv': revenue_by(valid.dropna(subset=
        ['team_code']), 'team_code'),
        'revenue_by_category.csv': revenue_by(valid, 'category'),
        'sales_after_results.csv': sales_after_results(valid, windows),
    }
    written = load(outputs, processed_dir)
    summary = {
        'total_rows': len(data['orders']),
        'duplicates': int((rejected['reason'] == 'duplicate').sum()),
        'valid_rows': len(valid),
        'rejected_rows': len(rejected),
        'total_revenue': round(float(valid['total'].sum()), 2),
        'files_written': len(written) + 1,
    }
    pd.DataFrame(list(summary.items()), columns=['metric',
    'value']).to_csv(
    os.path.join(processed_dir, 'summary.csv'), index=False)
    return summary